# **DATA LOADING**

Imports

In [2]:
import requests
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta

print("Libraries loaded!")

Libraries loaded!


Choosing the historical period

In [3]:
# We want the last 1 year of data
end_date = pd.Timestamp.now().normalize()
start_date = end_date - pd.DateOffset(years=1)

print("Start:", start_date)
print("End:", end_date)

Start: 2025-08-26 00:00:00
End: 2026-08-26 00:00:00


Testing the available timestamps

In [4]:
index_url = "https://www.smard.de/app/chart_data/410/DE/index_hour.json"

response = requests.get(index_url)
response.raise_for_status()

timestamps = response.json()["timestamps"]

timestamps = pd.to_datetime(
    timestamps,
    unit="ms"
)

print("Available data chunks:", len(timestamps))
print("First available:", timestamps.min())
print("Latest available:", timestamps.max())

Available data chunks: 609
First available: 2014-12-28 23:00:00
Latest available: 2026-08-23 22:00:00


Selecting only timestamps from our year

In [5]:
selected_timestamps = timestamps[
    (timestamps >= start_date) &
    (timestamps <= end_date)
]

print("Chunks selected:", len(selected_timestamps))
print("First chunk:", selected_timestamps.min())
print("Last chunk:", selected_timestamps.max())

Chunks selected: 52
First chunk: 2025-08-31 22:00:00
Last chunk: 2026-08-23 22:00:00


Create the historical downloader

In [6]:
def download_historical_data(filter_id, column_name):

    # Get available timestamps
    index_url = (
        f"https://www.smard.de/app/chart_data/"
        f"{filter_id}/DE/index_hour.json"
    )

    response = requests.get(index_url)
    response.raise_for_status()

    timestamps = response.json()["timestamps"]

    # If no timestamps are available
    if not timestamps:
        print(f"No data available for {column_name}")
        return pd.DataFrame(
            columns=["timestamp", column_name]
        )

    timestamps = pd.to_datetime(
        timestamps,
        unit="ms"
    )

    # Keep only our selected period
    timestamps = timestamps[
        (timestamps >= start_date) &
        (timestamps <= end_date)
    ]

    # If no timestamps exist in our period
    if len(timestamps) == 0:
        print(
            f"No data available for {column_name} "
            f"during selected period."
        )

        return pd.DataFrame(
            columns=["timestamp", column_name]
        )

    all_data = []

    # Download each chunk
    for i, timestamp in enumerate(timestamps):

        timestamp_ms = int(
            timestamp.timestamp() * 1000
        )

        data_url = (
            f"https://www.smard.de/app/chart_data/"
            f"{filter_id}/DE/"
            f"{filter_id}_DE_hour_{timestamp_ms}.json"
        )

        response = requests.get(data_url)

        if response.status_code != 200:
            print(
                f"Skipping chunk {timestamp}"
            )
            continue

        series = response.json()["series"]

        if not series:
            continue

        chunk = pd.DataFrame(
            series,
            columns=["timestamp", column_name]
        )

        all_data.append(chunk)

        print(
            f"{column_name}: "
            f"{i + 1}/{len(timestamps)} chunks"
        )

    # If nothing was downloaded
    if len(all_data) == 0:
        print(f"No usable data found for {column_name}")

        return pd.DataFrame(
            columns=["timestamp", column_name]
        )

    # Combine chunks
    df = pd.concat(
        all_data,
        ignore_index=True
    )

    # Convert timestamps
    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        unit="ms"
    )

    # Remove duplicates
    df = df.drop_duplicates(
        subset="timestamp"
    )

    # Keep requested period
    df = df[
        (df["timestamp"] >= start_date) &
        (df["timestamp"] <= end_date)
    ]

    # Sort
    df = df.sort_values("timestamp")

    return df.reset_index(drop=True)

Download electricity consumption

In [7]:
consumption = download_historical_data(
    410,
    "consumption_mw"
)

print()
print("Consumption shape:", consumption.shape)
display(consumption.head())

consumption_mw: 1/52 chunks
consumption_mw: 2/52 chunks
consumption_mw: 3/52 chunks
consumption_mw: 4/52 chunks
consumption_mw: 5/52 chunks
consumption_mw: 6/52 chunks
consumption_mw: 7/52 chunks
consumption_mw: 8/52 chunks
consumption_mw: 9/52 chunks
consumption_mw: 10/52 chunks
consumption_mw: 11/52 chunks
consumption_mw: 12/52 chunks
consumption_mw: 13/52 chunks
consumption_mw: 14/52 chunks
consumption_mw: 15/52 chunks
consumption_mw: 16/52 chunks
consumption_mw: 17/52 chunks
consumption_mw: 18/52 chunks
consumption_mw: 19/52 chunks
consumption_mw: 20/52 chunks
consumption_mw: 21/52 chunks
consumption_mw: 22/52 chunks
consumption_mw: 23/52 chunks
consumption_mw: 24/52 chunks
consumption_mw: 25/52 chunks
consumption_mw: 26/52 chunks
consumption_mw: 27/52 chunks
consumption_mw: 28/52 chunks
consumption_mw: 29/52 chunks
consumption_mw: 30/52 chunks
consumption_mw: 31/52 chunks
consumption_mw: 32/52 chunks
consumption_mw: 33/52 chunks
consumption_mw: 34/52 chunks
consumption_mw: 35/52 c

,timestamp,consumption_mw
0,2025-08-31 22:00:00,39947.01
1,2025-08-31 23:00:00,39017.77
2,2025-09-01 00:00:00,37956.16
3,2025-09-01 01:00:00,37650.69
4,2025-09-01 02:00:00,38324.95


Download generation + price data

In [8]:
wind = download_historical_data(
    4067,
    "wind_onshore_mw"
)

solar = download_historical_data(
    4068,
    "solar_mw"
)

gas = download_historical_data(
    4071,
    "gas_mw"
)

coal = download_historical_data(
    4069,
    "coal_mw"
)

price = download_historical_data(
    4169,
    "price_eur_mwh"
)

print("All historical datasets downloaded!")

wind_onshore_mw: 1/52 chunks
wind_onshore_mw: 2/52 chunks
wind_onshore_mw: 3/52 chunks
wind_onshore_mw: 4/52 chunks
wind_onshore_mw: 5/52 chunks
wind_onshore_mw: 6/52 chunks
wind_onshore_mw: 7/52 chunks
wind_onshore_mw: 8/52 chunks
wind_onshore_mw: 9/52 chunks
wind_onshore_mw: 10/52 chunks
wind_onshore_mw: 11/52 chunks
wind_onshore_mw: 12/52 chunks
wind_onshore_mw: 13/52 chunks
wind_onshore_mw: 14/52 chunks
wind_onshore_mw: 15/52 chunks
wind_onshore_mw: 16/52 chunks
wind_onshore_mw: 17/52 chunks
wind_onshore_mw: 18/52 chunks
wind_onshore_mw: 19/52 chunks
wind_onshore_mw: 20/52 chunks
wind_onshore_mw: 21/52 chunks
wind_onshore_mw: 22/52 chunks
wind_onshore_mw: 23/52 chunks
wind_onshore_mw: 24/52 chunks
wind_onshore_mw: 25/52 chunks
wind_onshore_mw: 26/52 chunks
wind_onshore_mw: 27/52 chunks
wind_onshore_mw: 28/52 chunks
wind_onshore_mw: 29/52 chunks
wind_onshore_mw: 30/52 chunks
wind_onshore_mw: 31/52 chunks
wind_onshore_mw: 32/52 chunks
wind_onshore_mw: 33/52 chunks
wind_onshore_mw: 34

Combine all datasets

In [9]:
df = consumption.copy()

df = df.merge(
    wind,
    on="timestamp",
    how="outer"
)

df = df.merge(
    solar,
    on="timestamp",
    how="outer"
)

df = df.merge(
    gas,
    on="timestamp",
    how="outer"
)

df = df.merge(
    coal,
    on="timestamp",
    how="outer"
)

df = df.merge(
    price,
    on="timestamp",
    how="outer"
)

df = df.sort_values("timestamp")
df = df.drop_duplicates("timestamp")
df = df.reset_index(drop=True)

print("Final shape:", df.shape)

display(df.head())

Final shape: (8619, 7)


,timestamp,consumption_mw,wind_onshore_mw,solar_mw,gas_mw,coal_mw,price_eur_mwh
0,2025-08-31 22:00:00,39947.01,12740.28,7.38,4596.50,1664.75,84.08
1,2025-08-31 23:00:00,39017.77,13182.18,7.49,4538.50,1616.25,81.43
2,2025-09-01 00:00:00,37956.16,12593.54,8.50,4705.75,1627.25,80.97
3,2025-09-01 01:00:00,37650.69,12599.03,7.91,4701.50,1632.00,80.03
4,2025-09-01 02:00:00,38324.95,12846.25,7.47,4962.75,1725.00,81.37


Basic data quality check

In [10]:
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nDate range:")
print(df["timestamp"].min())
print(df["timestamp"].max())

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate timestamps:")
print(df["timestamp"].duplicated().sum())

Rows: 8619
Columns: 7

Date range:
2025-08-31 22:00:00
2026-08-26 00:00:00

Missing values:
timestamp          0
consumption_mw     0
wind_onshore_mw    0
solar_mw           0
gas_mw             0
coal_mw            0
price_eur_mwh      0
dtype: int64

Duplicate timestamps:
0


save the actual dataset

In [11]:
# Create folders
Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("data/processed").mkdir(parents=True, exist_ok=True)

# Save raw historical data
df.to_parquet(
    "data/raw/smard_1year_raw.parquet",
    index=False
)

# Basic cleaned version
clean_df = df.copy()

clean_df = clean_df.sort_values("timestamp")
clean_df = clean_df.drop_duplicates("timestamp")
clean_df = clean_df.reset_index(drop=True)

clean_df.to_parquet(
    "data/processed/smard_1year_clean.parquet",
    index=False
)

print("================================")
print("Historical dataset saved!")
print("Rows:", len(clean_df))
print("Columns:", len(clean_df.columns))
print("================================")

Historical dataset saved!
Rows: 8619
Columns: 7


# **DATA CLEANING**

Load the saved dataset

In [12]:
import pandas as pd
import numpy as np

df = pd.read_parquet(
    "data/raw/smard_1year_raw.parquet"
)

print("Dataset loaded!")
print("Shape:", df.shape)

Dataset loaded!
Shape: (8619, 7)


In [13]:
print(df.dtypes)

timestamp          datetime64[ns]
consumption_mw            float64
wind_onshore_mw           float64
solar_mw                  float64
gas_mw                    float64
coal_mw                   float64
price_eur_mwh             float64
dtype: object


Check the time intervals

In [14]:
time_diff = df["timestamp"].diff()

print("Most common time interval:")
print(time_diff.value_counts().head())

Most common time interval:
timestamp
0 days 01:00:00    8618
Name: count, dtype: int64


Find missing hours

In [15]:
expected_hours = pd.date_range(
    start=df["timestamp"].min(),
    end=df["timestamp"].max(),
    freq="h"
)

actual_hours = df["timestamp"]

missing_hours = expected_hours.difference(actual_hours)

print("Expected hours:", len(expected_hours))
print("Actual rows:", len(actual_hours))
print("Missing hours:", len(missing_hours))

if len(missing_hours) > 0:
    print("\nFirst missing hours:")
    print(missing_hours[:10])

Expected hours: 8619
Actual rows: 8619
Missing hours: 0


Check negative generation values

In [16]:
generation_columns = [
    "consumption_mw",
    "wind_onshore_mw",
    "solar_mw",
    "gas_mw",
    "coal_mw"
]

for column in generation_columns:
    negative_count = (df[column] < 0).sum()
    print(column, "negative values:", negative_count)

consumption_mw negative values: 0
wind_onshore_mw negative values: 0
solar_mw negative values: 0
gas_mw negative values: 0
coal_mw negative values: 0


Check negative electricity prices

In [17]:
negative_prices = df[df["price_eur_mwh"] < 0]

print("Negative-price hours:", len(negative_prices))

print("\nLowest price:")
print(df["price_eur_mwh"].min())

print("\nHighest price:")
print(df["price_eur_mwh"].max())

Negative-price hours: 514

Lowest price:
-499.0

Highest price:
665.82


Basic statistics

In [18]:
print(df.describe())

                 timestamp  consumption_mw  wind_onshore_mw      solar_mw  \
count                 8619     8619.000000      8619.000000   8619.000000   
mean   2026-02-27 11:00:00    53853.215329     12981.167256   9190.435704   
min    2025-08-31 22:00:00    33233.270000       131.330000      0.000000   
25%    2025-11-29 16:30:00    46259.160000      5285.290000     11.640000   
50%    2026-02-27 11:00:00    53847.090000     10993.980000    290.810000   
75%    2026-05-28 05:30:00    60131.260000     18529.845000  14356.335000   
max    2026-08-26 00:00:00    78241.200000     46140.310000  58055.530000   
std                    NaN     9210.068853      9436.819437  14036.368785   

             gas_mw      coal_mw  price_eur_mwh  
count   8619.000000  8619.000000    8619.000000  
mean    7137.223835  3408.391411      98.741433  
min     1009.680000   135.190000    -499.000000  
25%     3627.385000  1834.465000      76.570000  
50%     6007.780000  3313.300000     100.620000  
75%   

Create a cleaner dataset

In [19]:
clean_df = df.copy()

# Sort by timestamp
clean_df = clean_df.sort_values("timestamp")

# Remove duplicate timestamps
clean_df = clean_df.drop_duplicates(
    subset="timestamp"
)

# Reset the index
clean_df = clean_df.reset_index(drop=True)

# Make sure all measurement columns are numeric
numeric_columns = [
    "consumption_mw",
    "wind_onshore_mw",
    "solar_mw",
    "gas_mw",
    "coal_mw",
    "price_eur_mwh"
]

for column in numeric_columns:
    clean_df[column] = pd.to_numeric(
        clean_df[column],
        errors="coerce"
    )

print("Clean dataset shape:", clean_df.shape)

Clean dataset shape: (8619, 7)


Final validation

In [20]:
print("Missing values:")
print(clean_df.isna().sum())

print("\nDuplicate timestamps:")
print(
    clean_df["timestamp"].duplicated().sum()
)

print("\nData types:")
print(clean_df.dtypes)

print("\nDate range:")
print(clean_df["timestamp"].min())
print(clean_df["timestamp"].max())

Missing values:
timestamp          0
consumption_mw     0
wind_onshore_mw    0
solar_mw           0
gas_mw             0
coal_mw            0
price_eur_mwh      0
dtype: int64

Duplicate timestamps:
0

Data types:
timestamp          datetime64[ns]
consumption_mw            float64
wind_onshore_mw           float64
solar_mw                  float64
gas_mw                    float64
coal_mw                   float64
price_eur_mwh             float64
dtype: object

Date range:
2025-08-31 22:00:00
2026-08-26 00:00:00


Save the cleaned dataset

In [21]:
output_path = "data/processed/smard_1year_clean.parquet"

clean_df.to_parquet(
    output_path,
    index=False
)

print("================================")
print("Clean dataset saved!")
print("File:", output_path)
print("Rows:", len(clean_df))
print("Columns:", len(clean_df.columns))
print("================================")

Clean dataset saved!
File: data/processed/smard_1year_clean.parquet
Rows: 8619
Columns: 7


# **ADDING MORE SOURCES**

Download the additional energy sources

In [22]:
# Download additional energy sources

offshore_wind = download_historical_data(
    1225,
    "wind_offshore_mw"
)

biomass = download_historical_data(
    4066,
    "biomass_mw"
)

hydro = download_historical_data(
    1226,
    "hydro_mw"
)

lignite = download_historical_data(
    1223,
    "lignite_mw"
)

other_renewables = download_historical_data(
    1228,
    "other_renewables_mw"
)

# Nuclear is skipped because there is no usable
# nuclear data in our selected historical period.
nuclear = pd.DataFrame(
    columns=["timestamp", "nuclear_mw"]
)

print("Additional datasets downloaded!")
print("Nuclear data skipped for this period.")

wind_offshore_mw: 1/52 chunks
wind_offshore_mw: 2/52 chunks
wind_offshore_mw: 3/52 chunks
wind_offshore_mw: 4/52 chunks
wind_offshore_mw: 5/52 chunks
wind_offshore_mw: 6/52 chunks
wind_offshore_mw: 7/52 chunks
wind_offshore_mw: 8/52 chunks
wind_offshore_mw: 9/52 chunks
wind_offshore_mw: 10/52 chunks
wind_offshore_mw: 11/52 chunks
wind_offshore_mw: 12/52 chunks
wind_offshore_mw: 13/52 chunks
wind_offshore_mw: 14/52 chunks
wind_offshore_mw: 15/52 chunks
wind_offshore_mw: 16/52 chunks
wind_offshore_mw: 17/52 chunks
wind_offshore_mw: 18/52 chunks
wind_offshore_mw: 19/52 chunks
wind_offshore_mw: 20/52 chunks
wind_offshore_mw: 21/52 chunks
wind_offshore_mw: 22/52 chunks
wind_offshore_mw: 23/52 chunks
wind_offshore_mw: 24/52 chunks
wind_offshore_mw: 25/52 chunks
wind_offshore_mw: 26/52 chunks
wind_offshore_mw: 27/52 chunks
wind_offshore_mw: 28/52 chunks
wind_offshore_mw: 29/52 chunks
wind_offshore_mw: 30/52 chunks
wind_offshore_mw: 31/52 chunks
wind_offshore_mw: 32/52 chunks
wind_offshore_mw:

Combine the additional datasets

In [23]:
df_full = clean_df.copy()

# Add the new energy sources
datasets = [
    offshore_wind,
    biomass,
    hydro,
    lignite,
    other_renewables
]

for new_data in datasets:
    df_full = df_full.merge(
        new_data,
        on="timestamp",
        how="left"
    )

# Sort by time
df_full = df_full.sort_values("timestamp")

# Remove duplicate timestamps
df_full = df_full.drop_duplicates(
    subset="timestamp"
)

# Reset index
df_full = df_full.reset_index(drop=True)

print("Final shape:", df_full.shape)

print("\nColumns:")
print(df_full.columns.tolist())

display(df_full.head())

Final shape: (8619, 12)

Columns:
['timestamp', 'consumption_mw', 'wind_onshore_mw', 'solar_mw', 'gas_mw', 'coal_mw', 'price_eur_mwh', 'wind_offshore_mw', 'biomass_mw', 'hydro_mw', 'lignite_mw', 'other_renewables_mw']


,timestamp,consumption_mw,wind_onshore_mw,solar_mw,gas_mw,coal_mw,price_eur_mwh,wind_offshore_mw,biomass_mw,hydro_mw,lignite_mw,other_renewables_mw
0,2025-08-31 22:00:00,39947.01,12740.28,7.38,4596.50,1664.75,84.08,4774.56,3552.27,1632.67,7031.50,100.30
1,2025-08-31 23:00:00,39017.77,13182.18,7.49,4538.50,1616.25,81.43,3709.63,3478.34,1622.75,6791.75,100.94
2,2025-09-01 00:00:00,37956.16,12593.54,8.50,4705.75,1627.25,80.97,2375.40,3455.61,1632.35,6920.00,101.41
3,2025-09-01 01:00:00,37650.69,12599.03,7.91,4701.50,1632.00,80.03,1490.00,3467.22,1545.54,7185.50,101.57
4,2025-09-01 02:00:00,38324.95,12846.25,7.47,4962.75,1725.00,81.37,765.38,3537.74,1556.14,7308.50,101.49


Check missing values

In [24]:
print("Missing values in each column:")
print(df_full.isna().sum())

Missing values in each column:
timestamp              0
consumption_mw         0
wind_onshore_mw        0
solar_mw               0
gas_mw                 0
coal_mw                0
price_eur_mwh          0
wind_offshore_mw       0
biomass_mw             0
hydro_mw               0
lignite_mw             0
other_renewables_mw    0
dtype: int64


Check exactly how many rows are missing

In [25]:
new_columns = [
    "wind_offshore_mw",
    "biomass_mw",
    "hydro_mw",
    "lignite_mw",
    "other_renewables_mw"
]

for column in new_columns:
    missing = df_full[column].isna().sum()
    print(f"{column}: {missing} missing values")

wind_offshore_mw: 0 missing values
biomass_mw: 0 missing values
hydro_mw: 0 missing values
lignite_mw: 0 missing values
other_renewables_mw: 0 missing values


Check the new data

In [26]:
print(
    df_full[
        new_columns
    ].describe().T
)

                      count         mean          std      min       25%  \
wind_offshore_mw     8619.0  3407.616509  2311.639137     0.67  1272.200   
biomass_mw           8619.0  4052.078138   394.259873  2901.55  3758.535   
hydro_mw             8619.0  1506.564810   305.552147   905.54  1294.330   
lignite_mw           8619.0  7473.527870  2946.803434  1357.37  5191.210   
other_renewables_mw  8619.0   103.498923    11.770916    67.73    92.520   

                         50%       75%       max  
wind_offshore_mw     3131.46  5444.630   8448.34  
biomass_mw           4007.40  4322.890   5071.75  
hydro_mw             1473.62  1681.265   2648.13  
lignite_mw           7623.71  9834.980  13661.82  
other_renewables_mw   106.43   112.235    127.48  


Total renewable generation

In [27]:
renewable_columns = [
    "wind_onshore_mw",
    "wind_offshore_mw",
    "solar_mw",
    "biomass_mw",
    "hydro_mw",
    "other_renewables_mw"
]

df_full["renewable_generation_mw"] = (
    df_full[renewable_columns].sum(axis=1)
)

print("Renewable generation feature created!")

display(
    df_full[
        [
            "timestamp",
            "renewable_generation_mw"
        ]
    ].head()
)

Renewable generation feature created!


,timestamp,renewable_generation_mw
0,2025-08-31 22:00:00,22807.46
1,2025-08-31 23:00:00,22101.33
2,2025-09-01 00:00:00,20166.81
3,2025-09-01 01:00:00,19211.27
4,2025-09-01 02:00:00,18814.47


Renewable share

Renewable Share =
Renewable Generation / Electricity Consumption × 100

In [29]:
df_full["renewable_share_pct"] = (
    df_full["renewable_generation_mw"]
    / df_full["consumption_mw"]
    * 100
)

print("Renewable share feature created!")

display(
    df_full[
        [
            "timestamp",
            "consumption_mw",
            "renewable_generation_mw",
            "renewable_share_pct"
        ]
    ].head()
)

Renewable share feature created!


,timestamp,consumption_mw,renewable_generation_mw,renewable_share_pct
0,2025-08-31 22:00:00,39947.01,22807.46,57.094286
1,2025-08-31 23:00:00,39017.77,22101.33,56.644267
2,2025-09-01 00:00:00,37956.16,20166.81,53.131850
3,2025-09-01 01:00:00,37650.69,19211.27,51.025014
4,2025-09-01 02:00:00,38324.95,18814.47,49.091962


In [30]:
print(
    df_full["renewable_share_pct"].describe()
)

count    8619.000000
mean       58.603629
std        27.886054
min        12.888051
25%        36.411689
50%        53.627040
75%        76.857120
max       149.255896
Name: renewable_share_pct, dtype: float64


In [31]:
print(
    "Negative renewable generation:",
    (df_full["renewable_generation_mw"] < 0).sum()
)

print(
    "Negative renewable share:",
    (df_full["renewable_share_pct"] < 0).sum()
)

print(
    "Renewable share above 100%:",
    (df_full["renewable_share_pct"] > 100).sum()
)

Negative renewable generation: 0
Negative renewable share: 0
Renewable share above 100%: 916


In [32]:
final_path = "data/processed/smard_1year_enriched.parquet"

df_full.to_parquet(
    final_path,
    index=False
)

print("================================")
print("Enriched dataset saved!")
print("Rows:", len(df_full))
print("Columns:", len(df_full.columns))
print("File:", final_path)
print("================================")

Enriched dataset saved!
Rows: 8619
Columns: 14
File: data/processed/smard_1year_enriched.parquet
